In [ ]:
### document  dat structure
import os
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader

In [ ]:
doc = Document(
    page_content = "This is the main text content I am using to create RAG",
    metadata = {
        "source" : "example.txt",
        "pages": 1,
        "author":"Nisha Ojha",
        "date_created" : "2026-01-30"
    }
)

In [ ]:
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Nisha Ojha', 'date_created': '2026-01-30'}, page_content='This is the main tect content I am using to create RAG')

In [ ]:
##create a simple txt file
os.makedirs("../data/text_files", exist_ok =True)

In [ ]:
sample_texts = {
    "../data/text_files/python_intro.txt": """
Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity
and readability. Created by Guido van Rossum and first released in 1991, Python has
become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence,
and automation.
""",

    "../data/text_files/machine_learning.txt": """
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems
to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it
to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Machine learning is widely applied in recommendation systems, image recognition,
fraud detection, and natural language processing.
"""
}

In [ ]:
for filepath, content in sample_texts.items():
    with open (filepath, "w", encoding = "utf-8") as f:
        f.write(content)
print("Sample Text files created !")

Sample Text files created !


In [ ]:
#loader.load is used to load the file and read the content
loader = TextLoader("../data/text_files/python_intro.txt", encoding = "utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='\nPython Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity\nand readability. Created by Guido van Rossum and first released in 1991, Python has\nbecome one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence,\nand automation.\n')]


In [4]:
# Directory Loader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 1️⃣ Load PDFs first
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

pdf_documents = dir_loader.load()

print(f"Loaded {len(pdf_documents)} documents")

# 2️⃣ Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# 3️⃣ Now split
chunks = text_splitter.split_documents(pdf_documents)

print(f"Total chunks created: {len(chunks)}")

Loaded 8 documents
Total chunks created: 100


In [5]:
###embedding and vector store DB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid  ## every record we store into vector db will have some id over that
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity  ##available in scikit learn

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1037.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded ✅


In [6]:
##This class will be reposible for the embedding part, class EmbeddingManager creates a blueprint for manageing text embeddinga
class EmbeddingManager:
    """
    Handles document embedding generation using SentenceTransformer
    """

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):  ## we are using all-MiniLM-L6-v2 available in hugging face somewhere we get 384 dimension
        """
        Initialize the embedding manager
        """
        self.model_name = model_name  ##stores the model
        self.model = None   ##set the model to none
        self._load_model() ##load model to load the embedding model

    def _load_model(self):   ##This method loads the SentenceTransformer model
        """
        Load the SentenceTransformer model
        """
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:  ## takes the lists of strings, checks if the model is loaded, convert each text to numerical embeddings, shows progress while processing, returns embeddings as a numpy array
        """
        Generate embeddings for a list of texts
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:  ## Returns the size of each embedding vector, For example: 384 or 768 numbers per sentence, Useful when storing embeddings in databases or vector stores
        """
        Get the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("Model not loaded")

        return self.model.get_sentence_embedding_dimension()

In [7]:
##initialize the embedding manager
embedding_manager = EmbeddingManager()


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 887.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [8]:
import sys
print(sys.executable)

c:\Users\Admin\Desktop\Data Science\RAG Pipeline From Scratch-Data Ingestion to Vector DB Pipeline\.venv\Scripts\python.exe


In [9]:
import os
import uuid
import numpy as np
import chromadb


class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store
    """

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize ChromaDB client and collection
        """
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: list, embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create vector store instance
vectorstore = VectorStore()
print(vectorstore)
print("Constructor ran")


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
Constructor ran


In [10]:
chunks

[Document(metadata={'producer': 'macOS Version 13.3 (Build 22E252) Quartz PDFContext, AppendMode 1.1', 'creator': 'TeX', 'creationdate': "D:20250123064829Z00'00'", 'source': '..\\data\\pdf\\01.pdf', 'file_path': '..\\data\\pdf\\01.pdf', 'total_pages': 6, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20250123064913Z00'00'", 'trapped': '', 'modDate': "D:20250123064913Z00'00'", 'creationDate': "D:20250123064829Z00'00'", 'page': 0}, page_content='1\nIntroduction\nCS 189 / 289A\n[Spring 2025]\nMachine Learning\nJonathan Shewchuk\nhttps://people.eecs.berkeley.edu/⇠jrs/189/\nHomework 1 due next Wednesday.\nQuestions: Please use Ed Discussion, not email.\n[Ed Discussion has an option for private questions, but\nplease use public for most questions so other people can beneﬁt.]\nFor personal matters only, jrs@berkeley.edu\nDiscussion sections (Tue & Wed):\nAttend any section. [We’ll put up a list on Ed Discussion.]'),
 Document(metadata={'producer':

In [13]:
##Extract all the texts and generate embeddings
##Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['1\nIntroduction\nCS 189 / 289A\n[Spring 2025]\nMachine Learning\nJonathan Shewchuk\nhttps://people.eecs.berkeley.edu/⇠jrs/189/\nHomework 1 due next Wednesday.\nQuestions: Please use Ed Discussion, not email.\n[Ed Discussion has an option for private questions, but\nplease use public for most questions so other people can beneﬁt.]\nFor personal matters only, jrs@berkeley.edu\nDiscussion sections (Tue & Wed):\nAttend any section. [We’ll put up a list on Ed Discussion.]',
 'Discussion sections (Tue & Wed):\nAttend any section. [We’ll put up a list on Ed Discussion.]\n[We might have a few advanced sections, including research discussion or exam problem preparation.]\nSections start Tuesday. [Next week.]\n[Enrollment: 736 students max. 349 waitlisted. Expecting many drops. EECS grads have highest priority;\nCD/DS undergrads second; non-EECS grads third; a few concurrent enrollment students will be admitted.]\n[Textbooks: Available free online. Linked from class web page.]',
 '[Textbooks: 